In [29]:
import torch
import torchvision
import torch.nn as nn
import torchvision.transforms as transforms
import pandas as pd
import numpy as np
import torch.nn.functional as F
import random
import matplotlib.pyplot as plt

### Seed + device

In [30]:
seed = 200

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
elif torch.backends.mps.is_available():
    torch.mps.manual_seed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

### Sprawdzenie średniej i odchylenia dla każdego kanału

In [31]:
transform = transforms.Compose([transforms.ToTensor()])
dataset = torchvision.datasets.ImageFolder("train/", transform=transform)

all_samples = torch.stack([sample[0] for sample in dataset])
print(f"Data norm per channel: mean={all_samples.mean(axis=(0,2,3))} | std={all_samples.std(axis=(0,2,3))}")

Data norm per channel: mean=tensor([0.5204, 0.4950, 0.4381]) | std=tensor([0.2841, 0.2770, 0.2974])


### Transformacje na zbiorach

In [32]:
train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomCrop(64, 4),
        transforms.RandomRotation(16),
        transforms.ToTensor(),
        transforms.Normalize((0.5204, 0.4950, 0.4381), (0.2841, 0.2770, 0.2974))])

val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5204, 0.4950, 0.4381), (0.2841, 0.2770, 0.2974))])

train_dataset = torchvision.datasets.ImageFolder("train/", transform=train_transform)
val_dataset = torchvision.datasets.ImageFolder("train/", transform=val_transform)

dataset_size = len(train_dataset)
train_size = int(0.9 * dataset_size)
val_size = dataset_size - train_size

indices = torch.randperm(dataset_size)

train_subset = torch.utils.data.Subset(train_dataset, indices[:train_size])
val_subset = torch.utils.data.Subset(val_dataset, indices[train_size:])

print(f"Train: {len(train_subset)}")
print(f"Val: {len(val_subset)}")


Train: 79209
Val: 8802


### Dataloadery do treningu i walidacji

In [34]:
train_loader = torch.utils.data.DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_subset, batch_size=32, shuffle=True)